In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.metrics import ndcg_score

# 1つ上のディレクトリにパスを通す
sys.path.append(os.path.abspath('../'))

from module.Dataset import read_preprocess_tsv, read_test_tsv, train_division, ColumnEncode


In [ ]:
# 前処理済みデータの読み込み
train_A = read_preprocess_tsv('A')
train_B = read_preprocess_tsv('B')
train_C = read_preprocess_tsv('C')
train_D = read_preprocess_tsv('D')

train = pd.concat([train_A, train_B, train_C, train_D], ignore_index=True)


In [ ]:
# 予測用データの読み込み
test = pd.read_csv('../dataset/test.tsv', sep='\t')


In [ ]:
# エンコードクラスのインスタンス化
user_item_enc = ColumnEncode()

In [ ]:
# エンコードの実行
train['user_id'] = user_item_enc.user_encode(train['user_id'])
train['product_id'] = user_item_enc.product_encode(train['product_id'])

test['user_id'] = user_item_enc.test_encode(test['user_id'])

In [ ]:
# タイムスタンプを数値型に変換
#train["time_stamp"] = pd.to_datetime(train["time_stamp"])
#train['unix_time'] = train['time_stamp'].astype(np.int64) // 10**9  # UNIX時間に変換


In [ ]:
# データ分割
train_X, train_y, valid_X, valid_y, features = train_division(train)


In [ ]:
# 時系列交差っぽい検証
model = xgb.XGBRegressor(objective='reg:squarederror')

model.fit(train_X, train_y)
predictions = model.predict(valid_X)
print("Validation NDCG:", ndcg_score([valid_y], [predictions]))


In [ ]:
# 最終学習用データ作成
X_train = train[features]
y_train = train['relevance']


In [ ]:
# 最終学習
model.fit(X_train, y_train)


In [ ]:
# 推薦関数
def recommend_products(user_id, top_k=22):
    user_data = train[train['user_id'] == user_id][features]
    if user_data.empty:
        print(f"Warning: User {user_id} has no data.")
        return pd.DataFrame(columns=['product_id', 'rank'])
    predictions = model.predict(user_data)
    user_data['score'] = predictions
    user_data = user_data.drop_duplicates(subset=['product_id']) # 重複する product_id を削除
    recommendations = user_data.sort_values(by='score', ascending=False).head(top_k)
    recommendations['rank'] = range(1, len(recommendations) + 1)
    recommendations['user_id'] = user_item_enc.user_decode(recommendations['user_id'])
    recommendations['product_id'] = user_item_enc.product_decode(recommendations['product_id'])
    return recommendations[['user_id', 'product_id', 'rank']]


In [ ]:
# 推薦結果と実際の結果を取得
def evaluate_ndcg():
    y_true = []
    y_score = []

    for user_id in test['user_id'].unique():
        # 実際の関連度 (trainのuser_idに基づいて)
        actual = train[train['user_id'] == user_id].sort_values(by='relevance', ascending=False)['relevance'].values

        # 推薦結果（モデルの予測スコアを基に）
        pred_df = recommend_products(user_id, top_k=len(actual))  # 商品IDとスコアの DataFrame
        if pred_df.empty:
            continue
        pred_ranks = pred_df['rank'].values
        pred_scores = 1 / (pred_ranks + 1e-5)  # 小さいrankほど高スコア
        # 関連度と予測スコアをリストに追加
        y_true.append(actual)
        y_score.append(pred_scores)

    # NDCGを計算
    return np.mean([ndcg_score([t], [s]) for t, s in zip(y_true, y_score)]) if y_true else 0.0


In [ ]:
# 予測結果保存
def save_predictions():
    results = []
    for user_id in test['user_id'].unique():
        if user_id in train['user_id'].values:
            recommendations = recommend_products(user_id)
            for _, row in recommendations.iterrows():
                results.append([row['user_id'], row['product_id'], row['rank']])
    if not results:
        print("Warning: No predictions generated.")
    pd.DataFrame(results, columns=['user_id', 'product_id', 'rank']).to_csv('../dataset/prediction_ALL.tsv', sep='\t', index=False)


In [ ]:
# 推薦結果表示
#print("nDCG Score:", evaluate_ndcg())


In [ ]:
save_predictions()
